In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import sqlite3
from sqlalchemy import create_engine
import os

BASE      = '/content/drive/MyDrive/ecommerce-etl-pipeline'
PROCESSED = f'{BASE}/data/processed'
SQL_PATH  = f'{BASE}/data/sql'

os.makedirs(SQL_PATH, exist_ok=True)
print("Ready!")

Mounted at /content/drive
Ready!


In [2]:
master    = pd.read_csv(f'{PROCESSED}/master_orders.csv')
customers = pd.read_csv(f'{PROCESSED}/customers_clean.csv')
products  = pd.read_csv(f'{PROCESSED}/products_clean.csv')
sellers   = pd.read_csv(f'{PROCESSED}/sellers_clean.csv')
payments  = pd.read_csv(f'{PROCESSED}/payments_clean.csv')
reviews   = pd.read_csv(f'{PROCESSED}/reviews_clean.csv')

print("All clean files loaded!")
print("Master shape:", master.shape)

All clean files loaded!
Master shape: (96470, 26)


In [3]:
# Create SQLite database
DB_PATH = f'{SQL_PATH}/ecommerce.db'
engine  = create_engine(f'sqlite:///{DB_PATH}')

print("Database created at:", DB_PATH)

Database created at: /content/drive/MyDrive/ecommerce-etl-pipeline/data/sql/ecommerce.db


In [4]:
# Load each table into the database
master.to_sql('master_orders', engine,
              if_exists='replace', index=False)

customers.to_sql('customers', engine,
                 if_exists='replace', index=False)

products.to_sql('products', engine,
                if_exists='replace', index=False)

sellers.to_sql('sellers', engine,
               if_exists='replace', index=False)

payments.to_sql('payments', engine,
                if_exists='replace', index=False)

reviews.to_sql('reviews', engine,
               if_exists='replace', index=False)

print("All tables loaded into database!")

All tables loaded into database!


In [5]:
# Connect and check all tables
conn = sqlite3.connect(DB_PATH)

query = """
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
"""

tables = pd.read_sql(query, conn)
print("Tables in database:")
print(tables)

Tables in database:
            name
0      customers
1  master_orders
2       payments
3       products
4        reviews
5        sellers


In [6]:
# Check row count of each table
for table in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as rows FROM {table}", conn)
    print(f"{table}: {count['rows'][0]:,} rows")

customers: 99,441 rows
master_orders: 96,470 rows
payments: 103,886 rows
products: 32,951 rows
reviews: 99,224 rows
sellers: 3,095 rows


In [7]:
query = """
SELECT
    order_status,
    COUNT(*) as total_orders,
    ROUND(SUM(total_revenue), 2) as total_revenue
FROM master_orders
GROUP BY order_status
ORDER BY total_orders DESC;
"""

result = pd.read_sql(query, conn)
print(result)

  order_status  total_orders  total_revenue
0    delivered         96470    15421082.85


In [10]:
query_1 = """
SELECT
    customer_state,
    COUNT(*) as total_orders
FROM master_orders
GROUP BY customer_state
ORDER BY total_orders DESC
LIMIT 5;
"""

result_1 = pd.read_sql(query_1, conn)
print("--- Top 5 Customer States ---")
print(result_1)

--- Top 5 Customer States ---
  customer_state  total_orders
0             SP         40494
1             RJ         12350
2             MG         11354
3             RS          5344
4             PR          4923


In [11]:
query_2 = """
SELECT
    ROUND(AVG(delivery_days), 2) as avg_delivery_time_days
FROM master_orders;
"""

result_2 = pd.read_sql(query_2, conn)
print("\n--- Average Delivery Time (Days) ---")
print(result_2)


--- Average Delivery Time (Days) ---
   avg_delivery_time_days
0                   12.09


In [12]:
query_3 = """
SELECT
    order_year,
    ROUND(SUM(total_revenue), 2) as total_revenue
FROM master_orders
GROUP BY order_year
ORDER BY order_year DESC;
"""

result_3 = pd.read_sql(query_3, conn)
print("\n--- Total Revenue by Year ---")
print(result_3)


--- Total Revenue by Year ---
   order_year  total_revenue
0        2018     8451925.11
1        2017     6922571.41
2        2016       46586.33


In [13]:
conn.close()
print("Database connection closed.")

Database connection closed.
